# Modify Forcing data

This notebook provides tools to load, modify and save forcing input (Simstrat-like `Forcing.dat`)

## Content:

- Load forcing data
- Modify forcing data
- Save modified forcing data

## Available functions:

- `data, units = load_forcing_data(path, file, start='file_start', stop='file_stop', load_units=True)` — Load forcing data and optionally load units of forcing variables
- `save_forcing_data(data, local_path, file_name, copy=True)` — Save modified forcing data to a new or a existing file
- `data = check_negative(data, val_save, var)` — Check for negative values for a forcing variable

---

## Import libraries

In [1]:
# Import standard Python libraries
import os
import numpy as np
import pandas as pd

# Import custom functions
from functions import load_forcing_data, save_forcing_data, check_negative

## Load forcing data

You can change these parameters:
- `local_path`: the path to the forcing data file
- `file_name`: the name of the forcing data file
    - usually `Forcing.dat`
- `pos_vars`: variables that cannot be negative
- `start`: from when on to edit forcing data
  - date and time in format `'YYYYMMDDHHMM'` (Year-Month-Day-Hour-Minute)
  - example: `'202510231500'` = October 23, 2025 at 15:00 (3:00 PM)
  - default: `'file_start'` -> start at beginning of file
- `stop`: until when to edit forcing data
  - same format as `start`
  - default: `'file_stop'` -> stop at end of file
- `load_units`: whether to load the units from the file
    - `True` to load the units (default)
    - `False` to not load the units, returns an empty DataFrame to `units`

In [2]:
# Set parameters
local_path = '../../tests/TestCase_LakeConstance'
file_name = 'MeteoForcing.dat'
pos_vars = ['sol', 'vap', 'cloud', 'rain']

# Load data
data, units = load_forcing_data(
    path = local_path,
    file = file_name,
    start = 'file_start',
    stop = 'file_stop',
    load_units = True
)

# Display units and data
display(units)
display(data.round(4))

Loading data from local file: ../../tests/TestCase_LakeConstance\MeteoForcing.dat
✓ Loaded 380952 rows from ../../tests/TestCase_LakeConstance\MeteoForcing.dat
✓ Filtered to date range: 380952 rows between 1981-01-01 and 2024-06-16


,u,v,Tair,sol,vap,cloud,rain
Unit,m/s,m/s,°C,W/m2,mbar,-,m/hr


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,1.2879,0.3762,2.6679,0.0000,5.1902,0.7361,0.0
1,1981-01-01 01:00:02,2.6807,0.3587,3.1668,0.0000,5.5153,0.6797,0.0
2,1981-01-01 01:59:57,4.7271,1.8239,2.3686,0.0000,5.5153,0.7023,0.0
3,1981-01-01 03:00:00,5.0125,1.5562,2.9673,0.0000,5.1902,0.7178,0.0
4,1981-01-01 04:00:02,6.1014,1.3453,3.4662,1.7028,4.6482,0.7250,0.0
...,...,...,...,...,...,...,...,...
380947,2024-06-16 19:00:02,-0.4474,1.0049,20.4195,38.0000,15.6000,0.1258,0.0
380948,2024-06-16 19:59:57,-0.2088,1.4854,20.0195,6.0000,14.6000,0.0983,0.0
380949,2024-06-16 21:00:00,0.7607,1.6314,19.1195,1.0000,14.8000,0.0881,0.0
380950,2024-06-16 22:00:02,0.7211,1.2000,17.4195,0.0000,15.0000,0.0917,0.0


## Modify forcing data

You can modify the data in the following ways:
- `Add a constant`
- `Multiply by a constant`
- `Set to a constant`
- `Add a linear trend`

Be aware that:
- The variables in pos_vars cannot be negative
    - this is controlled in this code

### Add constant

You can change these parameters:
- `variable`: choose the variable to modify from:
    - `'u'`, `'v'`, `'Tair'`, `'sol'`, `'vap'`, `'cloud'`, `'rain'`
- `constant`: the constant to add
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative

In [3]:
# Set parameters
variable = 'u'
constant = -1

# Modify and display data
val_save = data[variable].copy()
data[variable] += float(constant)
if variable in pos_vars:
    data[variable] = check_negative(
    data[variable], val_save, variable)
else:
    print(f"✓ Forcing variable '{variable}' modified")
display(data.round(4))

✓ Forcing variable 'u' modified


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,0.2879,0.3762,2.6679,0.0000,5.1902,0.7361,0.0
1,1981-01-01 01:00:02,1.6807,0.3587,3.1668,0.0000,5.5153,0.6797,0.0
2,1981-01-01 01:59:57,3.7271,1.8239,2.3686,0.0000,5.5153,0.7023,0.0
3,1981-01-01 03:00:00,4.0125,1.5562,2.9673,0.0000,5.1902,0.7178,0.0
4,1981-01-01 04:00:02,5.1014,1.3453,3.4662,1.7028,4.6482,0.7250,0.0
...,...,...,...,...,...,...,...,...
380947,2024-06-16 19:00:02,-1.4474,1.0049,20.4195,38.0000,15.6000,0.1258,0.0
380948,2024-06-16 19:59:57,-1.2088,1.4854,20.0195,6.0000,14.6000,0.0983,0.0
380949,2024-06-16 21:00:00,-0.2393,1.6314,19.1195,1.0000,14.8000,0.0881,0.0
380950,2024-06-16 22:00:02,-0.2789,1.2000,17.4195,0.0000,15.0000,0.0917,0.0


### Multiply by constant

You can change these parameters:
- `variable`: choose the variable to modify from:
    - `'u'`, `'v'`, `'Tair'`, `'sol'`, `'vap'`, `'cloud'`, `'rain'`
- `constant`: the constant to multiply by
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative

In [4]:
# Set parameters
variable = 'v'
constant = 1/10

# Modify and display data
val_save = data[variable].copy()
data[variable] *= float(constant)
if variable in pos_vars:
    data[variable] = check_negative(
    data[variable], val_save, variable)
else:
    print(f"✓ Forcing variable '{variable}' modified")
display(data.round(4))

✓ Forcing variable 'v' modified


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,0.2879,0.0376,2.6679,0.0000,5.1902,0.7361,0.0
1,1981-01-01 01:00:02,1.6807,0.0359,3.1668,0.0000,5.5153,0.6797,0.0
2,1981-01-01 01:59:57,3.7271,0.1824,2.3686,0.0000,5.5153,0.7023,0.0
3,1981-01-01 03:00:00,4.0125,0.1556,2.9673,0.0000,5.1902,0.7178,0.0
4,1981-01-01 04:00:02,5.1014,0.1345,3.4662,1.7028,4.6482,0.7250,0.0
...,...,...,...,...,...,...,...,...
380947,2024-06-16 19:00:02,-1.4474,0.1005,20.4195,38.0000,15.6000,0.1258,0.0
380948,2024-06-16 19:59:57,-1.2088,0.1485,20.0195,6.0000,14.6000,0.0983,0.0
380949,2024-06-16 21:00:00,-0.2393,0.1631,19.1195,1.0000,14.8000,0.0881,0.0
380950,2024-06-16 22:00:02,-0.2789,0.1200,17.4195,0.0000,15.0000,0.0917,0.0


### Set to constant

You can change these parameters:
- `variable`: choose the variable to modify from:
    - `'u'`, `'v'`, `'Tair'`, `'sol'`, `'vap'`, `'cloud'`, `'rain'`
- `constant`: the constant value of the variable
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative

In [5]:
# Set parameters
variable = 'Tair'
constant = 10

# Modify and display data
val_save = data[variable].copy()
data[variable] = np.full((len(data)), float(constant))
if variable in pos_vars:
    data[variable] = check_negative(
    data[variable], val_save, variable)
else:
    print(f"✓ Forcing variable '{variable}' modified")
display(data.round(4))

✓ Forcing variable 'Tair' modified


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,0.2879,0.0376,10.0,0.0000,5.1902,0.7361,0.0
1,1981-01-01 01:00:02,1.6807,0.0359,10.0,0.0000,5.5153,0.6797,0.0
2,1981-01-01 01:59:57,3.7271,0.1824,10.0,0.0000,5.5153,0.7023,0.0
3,1981-01-01 03:00:00,4.0125,0.1556,10.0,0.0000,5.1902,0.7178,0.0
4,1981-01-01 04:00:02,5.1014,0.1345,10.0,1.7028,4.6482,0.7250,0.0
...,...,...,...,...,...,...,...,...
380947,2024-06-16 19:00:02,-1.4474,0.1005,10.0,38.0000,15.6000,0.1258,0.0
380948,2024-06-16 19:59:57,-1.2088,0.1485,10.0,6.0000,14.6000,0.0983,0.0
380949,2024-06-16 21:00:00,-0.2393,0.1631,10.0,1.0000,14.8000,0.0881,0.0
380950,2024-06-16 22:00:02,-0.2789,0.1200,10.0,0.0000,15.0000,0.0917,0.0


### Add linear trend

You can change these parameters:
- `variable`: choose the variable to modify from:
    - `'u'`, `'v'`, `'Tair'`, `'sol'`, `'vap'`, `'cloud'`, `'rain'`
- `step`: the time of change
    - any positive integer
    - usually `1` corresponds to one hour
    - accordingly `24` corresponds to one day
    -  `8760` corresponds to one year
- `slope`: the amount of change per time of change
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative
    - `variable` increases by `slope` every `step` hours
- `offset`: the offset to the trend
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative
    - default is `0`

In [6]:
# Set parameters
variable = 'Tair'
step = 8760
slope = 1/43
offset = 0

# Modify and display data
rate = float(slope / step)
val_save = data[variable].copy()
data[variable] = (data[variable] +
    np.linspace(0, rate * (len(data) - 1), len(data)) +
    float(offset))
if variable in pos_vars:
    data[variable] = check_negative(
    data[variable], val_save, variable)
else:
    print(f"✓ Forcing variable '{variable}' modified")
display(data.round(4))

✓ Forcing variable 'Tair' modified


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,0.2879,0.0376,10.0000,0.0000,5.1902,0.7361,0.0
1,1981-01-01 01:00:02,1.6807,0.0359,10.0000,0.0000,5.5153,0.6797,0.0
2,1981-01-01 01:59:57,3.7271,0.1824,10.0000,0.0000,5.5153,0.7023,0.0
3,1981-01-01 03:00:00,4.0125,0.1556,10.0000,0.0000,5.1902,0.7178,0.0
4,1981-01-01 04:00:02,5.1014,0.1345,10.0000,1.7028,4.6482,0.7250,0.0
...,...,...,...,...,...,...,...,...
380947,2024-06-16 19:00:02,-1.4474,0.1005,11.0113,38.0000,15.6000,0.1258,0.0
380948,2024-06-16 19:59:57,-1.2088,0.1485,11.0113,6.0000,14.6000,0.0983,0.0
380949,2024-06-16 21:00:00,-0.2393,0.1631,11.0113,1.0000,14.8000,0.0881,0.0
380950,2024-06-16 22:00:02,-0.2789,0.1200,11.0113,0.0000,15.0000,0.0917,0.0


# Step 4
## Save modified forcing data

You can change this parameter:
- `copy`: Wheter to create a new forcing file (with _modified appended to the file_name) or to overwrite the original
    - `True` to create a new file (default)
    - `False` to overwrite the original

In [7]:
# Save data
save_forcing_data(
    data,
    path = local_path,
    file = file_name,
    copy = True
)

Writing modified data into: ../../tests/TestCase_LakeConstance\MeteoForcing_modified.dat
✓ Modified 380952 rows
